# StyleFormer Training on Kaggle

Train face transformation models using StyleGAN latent space editing.

## Prerequisites
- Enable GPU: Settings → Accelerator → GPU T4 x2 or P100
- Add datasets:
  - `celeba-dataset` or `celebahq-resized-256x256`
  - `styleformer` (this codebase as a dataset)

## 1. Environment Setup

In [ ]:
# Install required packages
!pip install -q lpips einops pytorch-lightning hydra-core omegaconf wandb

In [ ]:
import sys
import os
from pathlib import Path

# Add StyleFormer to path
STYLEFORMER_PATH = Path('/kaggle/input/styleformer')
if not STYLEFORMER_PATH.exists():
    # Try alternative paths
    for alt in ['/kaggle/input/styleformer-code', '/kaggle/working/StyleFormer']:
        if Path(alt).exists():
            STYLEFORMER_PATH = Path(alt)
            break

sys.path.insert(0, str(STYLEFORMER_PATH))
print(f"StyleFormer path: {STYLEFORMER_PATH}")

# Setup Kaggle environment
from kaggle.setup_kaggle import setup, KaggleConfig
config = KaggleConfig()
setup_result = setup(install_packages=False)  # Already installed above

In [ ]:
# Verify imports
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

from src.data import FaceDataModule
from src.models.losses import L1Loss, PerceptualLoss, IdentityLoss
from src.evaluation import Evaluator
print("\nStyleFormer imports successful!")

## 2. Configuration

In [ ]:
# Training configuration
TRAIN_CONFIG = {
    # Data
    'dataset': 'celeba_hq',
    'image_size': 256,
    'batch_size': 8,
    'num_workers': 2,
    
    # Model
    'w_dim': 512,
    'num_ws': 14,  # For 256x256
    
    # Training
    'learning_rate': 1e-4,
    'max_epochs': 50,
    'val_check_interval': 0.25,
    
    # Losses
    'loss_weights': {
        'reconstruction': 1.0,
        'perceptual': 0.8,
        'identity': 0.1,
    },
    
    # Hardware
    'accelerator': config.get_accelerator(),
    'devices': 1,
    'precision': '16-mixed',  # Use mixed precision for speed
}

print("Training config:")
for k, v in TRAIN_CONFIG.items():
    print(f"  {k}: {v}")

## 3. Dataset Setup

In [ ]:
# Find CelebA dataset
celeba_paths = [
    '/kaggle/input/celeba-dataset',
    '/kaggle/input/celebahq-resized-256x256',
    '/kaggle/input/celeba-hq-resized',
]

DATASET_PATH = None
for path in celeba_paths:
    if Path(path).exists():
        DATASET_PATH = Path(path)
        break

if DATASET_PATH:
    print(f"Found dataset: {DATASET_PATH}")
    !ls -la {DATASET_PATH}
else:
    print("No dataset found! Please add one of:")
    for p in celeba_paths:
        print(f"  - {p}")

In [ ]:
# Create data module
if DATASET_PATH:
    datamodule = FaceDataModule(
        name=TRAIN_CONFIG['dataset'],
        root=str(DATASET_PATH),
        image_size=TRAIN_CONFIG['image_size'],
        batch_size=TRAIN_CONFIG['batch_size'],
        num_workers=TRAIN_CONFIG['num_workers'],
        selected_attrs=['Male', 'Young'],
    )
    
    # Setup and check
    datamodule.setup('fit')
    print(f"Train samples: {len(datamodule.train_dataset)}")
    print(f"Val samples: {len(datamodule.val_dataset)}")
    
    # Visualize a batch
    batch = next(iter(datamodule.train_dataloader()))
    print(f"Batch keys: {batch.keys()}")
    print(f"Image shape: {batch['image'].shape}")

## 4. Model Setup

In [ ]:
import torch.nn as nn
import pytorch_lightning as pl

from src.models.encoders.base import BaseEncoder, GradualStyleBlock
from src.models.generators.base import BaseGenerator, SynthesisLayer, ToRGB
from src.models.losses import L1Loss, PerceptualLoss
from src.models.lightning_modules.base import BaseTransformerModule


class SimpleEncoder(BaseEncoder):
    """Simple encoder for demonstration."""
    
    def __init__(self, w_dim=512, num_ws=14, input_size=256):
        super().__init__(w_dim=w_dim, num_ws=num_ws, input_size=input_size)
        
        # Simple CNN backbone
        self.backbone = nn.Sequential(
            nn.Conv2d(3, 64, 7, 2, 3),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 128, 3, 2, 1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 256, 3, 2, 1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 512, 3, 2, 1),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool2d(1),
        )
        
        # Style prediction layers
        self.styles = nn.ModuleList([
            nn.Linear(512, w_dim) for _ in range(num_ws)
        ])
    
    def forward(self, x):
        features = self.backbone(x).flatten(1)  # (B, 512)
        styles = [layer(features) for layer in self.styles]
        return torch.stack(styles, dim=1)  # (B, num_ws, w_dim)


class SimpleDecoder(nn.Module):
    """Simple decoder for demonstration."""
    
    def __init__(self, w_dim=512, output_size=256):
        super().__init__()
        self.w_dim = w_dim
        self.output_size = output_size
        
        # Constant input
        self.const = nn.Parameter(torch.randn(1, 512, 4, 4))
        
        # Upsampling layers
        self.layers = nn.ModuleList([
            nn.ConvTranspose2d(512, 256, 4, 2, 1),  # 8
            nn.ConvTranspose2d(256, 128, 4, 2, 1),  # 16
            nn.ConvTranspose2d(128, 64, 4, 2, 1),   # 32
            nn.ConvTranspose2d(64, 64, 4, 2, 1),    # 64
            nn.ConvTranspose2d(64, 32, 4, 2, 1),    # 128
            nn.ConvTranspose2d(32, 16, 4, 2, 1),    # 256
        ])
        
        self.to_rgb = nn.Conv2d(16, 3, 1)
        self.act = nn.LeakyReLU(0.2)
    
    def forward(self, w):
        # w: (B, num_ws, w_dim)
        x = self.const.repeat(w.shape[0], 1, 1, 1)
        
        for layer in self.layers:
            x = self.act(layer(x))
        
        return torch.tanh(self.to_rgb(x))


class FaceTransformer(BaseTransformerModule):
    """Face transformation model for training."""
    
    def __init__(self, **kwargs):
        super().__init__(**kwargs)
        
        # Initialize encoder and decoder
        self.encoder = SimpleEncoder(
            w_dim=self.w_dim,
            num_ws=self.num_ws,
        )
        self.decoder = SimpleDecoder(w_dim=self.w_dim)
        
        # Setup losses
        self.setup_losses()
    
    def setup_losses(self):
        self.losses['reconstruction'] = L1Loss()
        try:
            self.losses['perceptual'] = PerceptualLoss()
        except Exception as e:
            print(f"Warning: Could not load perceptual loss: {e}")
        
        self.loss_weights = {
            'reconstruction': 1.0,
            'perceptual': 0.8,
        }
    
    def encode(self, x):
        return self.encoder(x)
    
    def decode(self, w):
        return self.decoder(w)


print("Model classes defined!")

In [ ]:
# Create model
model = FaceTransformer(
    learning_rate=TRAIN_CONFIG['learning_rate'],
    w_dim=TRAIN_CONFIG['w_dim'],
    num_ws=TRAIN_CONFIG['num_ws'],
)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## 5. Training

In [ ]:
from pytorch_lightning import Trainer
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import WandbLogger, CSVLogger

# Callbacks
callbacks = [
    ModelCheckpoint(
        dirpath=config.checkpoint_dir,
        filename='styleformer-{epoch:02d}-{val_total_loss:.4f}',
        save_top_k=3,
        monitor='val/total_loss',
        mode='min',
    ),
    EarlyStopping(
        monitor='val/total_loss',
        patience=10,
        mode='min',
    ),
    LearningRateMonitor(logging_interval='step'),
]

# Logger (use CSV for Kaggle, or WandB if you have an account)
logger = CSVLogger(config.output_dir, name='styleformer')

# Trainer
trainer = Trainer(
    max_epochs=TRAIN_CONFIG['max_epochs'],
    accelerator=TRAIN_CONFIG['accelerator'],
    devices=TRAIN_CONFIG['devices'],
    precision=TRAIN_CONFIG['precision'],
    callbacks=callbacks,
    logger=logger,
    val_check_interval=TRAIN_CONFIG['val_check_interval'],
    log_every_n_steps=50,
    gradient_clip_val=1.0,
)

print(f"Trainer configured with {TRAIN_CONFIG['accelerator']}")

In [ ]:
# Train!
if DATASET_PATH:
    trainer.fit(model, datamodule)
else:
    print("No dataset available. Please add a CelebA dataset to continue.")

## 6. Evaluation

In [ ]:
# Load best checkpoint
if DATASET_PATH and trainer.checkpoint_callback.best_model_path:
    best_model = FaceTransformer.load_from_checkpoint(
        trainer.checkpoint_callback.best_model_path
    )
    best_model.eval()
    print(f"Loaded best model from: {trainer.checkpoint_callback.best_model_path}")

In [ ]:
# Generate sample outputs
import matplotlib.pyplot as plt
from src.data.transforms import denormalize

if DATASET_PATH:
    model.eval()
    
    # Get a batch
    batch = next(iter(datamodule.val_dataloader()))
    images = batch['image'][:8].to(config.get_device())
    
    # Encode and decode
    with torch.no_grad():
        latents = model.encode(images)
        reconstructed = model.decode(latents)
    
    # Visualize
    fig, axes = plt.subplots(2, 8, figsize=(16, 4))
    
    for i in range(8):
        # Original
        orig = denormalize(images[i]).permute(1, 2, 0).cpu().numpy()
        axes[0, i].imshow(orig.clip(0, 1))
        axes[0, i].axis('off')
        if i == 0:
            axes[0, i].set_title('Original')
        
        # Reconstructed
        recon = denormalize(reconstructed[i]).permute(1, 2, 0).cpu().numpy()
        axes[1, i].imshow(recon.clip(0, 1))
        axes[1, i].axis('off')
        if i == 0:
            axes[1, i].set_title('Reconstructed')
    
    plt.tight_layout()
    plt.savefig(config.output_dir / 'reconstruction_samples.png', dpi=150)
    plt.show()

## 7. Save Model for Kaggle Output

In [ ]:
# Save final model
if DATASET_PATH:
    output_path = config.output_dir / 'final_model.ckpt'
    trainer.save_checkpoint(output_path)
    print(f"Saved model to: {output_path}")
    
    # Also save as torch state dict for easier loading
    torch.save({
        'encoder': model.encoder.state_dict(),
        'decoder': model.decoder.state_dict(),
        'config': TRAIN_CONFIG,
    }, config.output_dir / 'model_weights.pt')
    print(f"Saved weights to: {config.output_dir / 'model_weights.pt'}")

In [ ]:
# List outputs
print("\nOutput files:")
!ls -la /kaggle/working/outputs/